# **MycoGuard: AI Based Mycofiltration Diagnostics**
This notebook contains the end-to-end pipeline for training a deep learning model to monitor the health of biological water filters.

In [ ]:
import subprocess, os
subprocess.run(['fusermount', '-u', '/content/drive'], capture_output=True)
from google.colab import drive
drive.mount('/content/drive')

**Directory Verification**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# **1. Data Acquisition**

In [ ]:
import os

# Check if zip exists in Drive
zip_path = "/content/drive/MyDrive/devpost-hackathon/MushroomDiseaseDataset(Healthy,Single Infected/Mushroom Disease Dataset (Healthy, Single infected & Mixed infected )"

if os.path.exists(zip_path):
    size_mb = os.path.getsize(zip_path) / (1024*1024)
    print(f"✓ Found dataset_root.zip ({size_mb:.1f} MB)")
else:
    print("✗ NOT FOUND — dataset_root.zip is not in your Google Drive root")
    print("\nFiles in your Drive root:")
    for f in os.listdir("/content/drive/MyDrive"):
        print(f"  {f}")

In [ ]:
import zipfile, os, shutil

zip_path = "/content/drive/MyDrive/dataset_root.zip"
extract_path = "/content/dataset_root"

if os.path.exists(extract_path):
    shutil.rmtree(extract_path)
    print("Removed old folder.")

print("Extracting... this will take 2-3 minutes...")
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall("/content/")

print("\nDone! Checking structure:")
for split in ["train", "val", "test"]:
    for cls in ["healthy", "degrading", "replace_now"]:
        path = f"/content/dataset_root/{split}/{cls}"
        if os.path.exists(path):
            count = len([f for f in os.listdir(path) if f.lower().endswith(('.jpg','.jpeg','.png','.bmp'))])
            print(f"  {split}/{cls}: {count} images")
        else:
            print(f"  {split}/{cls}: MISSING")

# **2. Visual Inspection**

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import os
import random

folder = "/content/dataset_root/train/healthy"

images = random.sample(os.listdir(folder), 3)

for img_name in images:
    img = Image.open(os.path.join(folder, img_name))
    plt.imshow(img)
    plt.title(img_name)
    plt.axis("off")
    plt.show()

# **3. Image Preprocessing**

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os

data_dir = "/content/dataset_root"

transform = {
    "train": transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomResizedCrop(224, scale=(0.75, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
        transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
    "val_test": transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ])
}

train_data = datasets.ImageFolder(os.path.join(data_dir, "train"), transform=transform["train"])
val_data   = datasets.ImageFolder(os.path.join(data_dir, "val"),   transform=transform["val_test"])
test_data  = datasets.ImageFolder(os.path.join(data_dir, "test"),  transform=transform["val_test"])

print("Classes:", train_data.classes)
print("Train:", len(train_data), "Val:", len(val_data), "Test:", len(test_data))


In [ ]:
data_dir = "/content/dataset_root"

# **4. Addressing Class Imbalance**

In [ ]:
from torch.utils.data import DataLoader
from collections import Counter
import torch

batch_size = 32

train_loader = DataLoader(
    train_data, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_data, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    test_data, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True
)

class_counts = Counter(label for _, label in train_data.samples)
print("Training class counts:")
for idx, class_name in enumerate(train_data.classes):
    print(f"  {class_name}: {class_counts[idx]}")

num_samples = len(train_data.samples)
num_classes = len(train_data.classes)
class_weights = torch.tensor(
    [num_samples / (num_classes * class_counts[i]) for i in range(num_classes)],
    dtype=torch.float
)
print("Computed class weights:", class_weights.tolist())


# **5. Training Logic**

In [ ]:
import copy
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchvision
from torch import nn, optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
model = torchvision.models.efficientnet_b0(weights='IMAGENET1K_V1')

# Freeze backbone first
for param in model.features.parameters():
    param.requires_grad = False

# Unfreeze the later feature blocks for domain adaptation
for param in model.features[6].parameters():
    param.requires_grad = True
for param in model.features[7].parameters():
    param.requires_grad = True
for param in model.features[8].parameters():
    param.requires_grad = True

in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(in_features, len(train_data.classes))
)

model = model.to(device)

criterion = nn.CrossEntropyLoss(
    weight=class_weights.to(device),
    label_smoothing=0.1
)

optimizer = optim.Adam([
    {"params": model.features[6].parameters(), "lr": 1e-5},
    {"params": model.features[7].parameters(), "lr": 1e-5},
    {"params": model.features[8].parameters(), "lr": 1e-5},
    {"params": model.classifier.parameters(), "lr": 1e-4},
], weight_decay=1e-4)

scheduler = ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=4, min_lr=1e-7
)

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total
    return epoch_loss, epoch_acc

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return epoch_loss, epoch_acc, macro_f1, np.array(all_labels), np.array(all_preds)

epochs = 20
best_val_acc = 0.0
best_state = None

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
    "val_f1": [],
    "lr": []
}

for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader, criterion, device)

    scheduler.step(val_acc)
    current_lr = optimizer.param_groups[-1]["lr"]

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_f1"].append(val_f1)
    history["lr"].append(current_lr)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())
        torch.save(best_state, "/content/mycoguard_best_efficientnet.pth")
        saved_flag = " <- SAVED"
    else:
        saved_flag = ""

    print(
        f"Epoch {epoch+1:02d}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% | "
        f"Val F1: {val_f1:.4f} | LR: {current_lr:.7f}{saved_flag}"
    )

print(f"\nTraining complete. Best validation accuracy: {best_val_acc:.2f}%")


In [ ]:
!pip install seaborn

# **6. Results Visualization**

In [ ]:
# Plot training curves
plt.figure(figsize=(7, 4))
plt.plot(history["train_acc"], label="Train Accuracy")
plt.plot(history["val_acc"], label="Val Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(history["train_loss"], label="Train Loss")
plt.plot(history["val_loss"], label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(history["val_f1"], label="Validation Macro F1")
plt.xlabel("Epoch")
plt.ylabel("Macro F1")
plt.title("Validation Macro F1 Across Epochs")
plt.legend()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(history["lr"], label="Learning Rate")
plt.xlabel("Epoch")
plt.ylabel("LR")
plt.title("Learning Rate Schedule")
plt.legend()
plt.show()


In [ ]:
model.load_state_dict(torch.load("/content/mycoguard_best_efficientnet.pth", map_location=device))

test_loss, test_acc, test_f1, y_true, y_pred = evaluate(model, test_loader, criterion, device)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%")
print(f"Test Macro F1: {test_f1:.4f}")

cm = confusion_matrix(y_true, y_pred)
print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=train_data.classes))


In [ ]:
import seaborn as sns

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt="d",
    xticklabels=train_data.classes,
    yticklabels=train_data.classes
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Test Confusion Matrix")
plt.show()


In [ ]:
inv_normalize = transforms.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
    std=[1/0.229, 1/0.224, 1/0.225]
)

model.eval()
images, labels = next(iter(test_loader))
images = images.to(device)
labels = labels.to(device)

with torch.no_grad():
    outputs = model(images)
    preds = outputs.argmax(dim=1)

plt.figure(figsize=(14, 10))
for i in range(min(12, len(images))):
    plt.subplot(3, 4, i+1)
    img = inv_normalize(images[i].cpu()).permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)
    plt.imshow(img)
    plt.title(f"True: {train_data.classes[labels[i].item()]}\nPred: {train_data.classes[preds[i].item()]}")
    plt.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
replace_idx = train_data.class_to_idx["replace_now"]
mistake_indices = np.where((y_true == replace_idx) & (y_pred != replace_idx))[0]

print(f"Misclassified 'replace_now' samples: {len(mistake_indices)}")

ordered_test_loader = DataLoader(
    test_data, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True
)

all_images = []
all_labels = []

for batch_images, batch_labels in ordered_test_loader:
    all_images.append(batch_images)
    all_labels.append(batch_labels)

all_images = torch.cat(all_images, dim=0)
all_labels = torch.cat(all_labels, dim=0)

plt.figure(figsize=(14, 10))
for plot_i, sample_idx in enumerate(mistake_indices[:12]):
    plt.subplot(3, 4, plot_i + 1)
    img = inv_normalize(all_images[sample_idx]).permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)
    true_name = train_data.classes[y_true[sample_idx]]
    pred_name = train_data.classes[y_pred[sample_idx]]
    plt.imshow(img)
    plt.title(f"True: {true_name}\nPred: {pred_name}")
    plt.axis("off")

plt.tight_layout()
plt.show()
